# 03 — Download Court documents

This notebook is the acquisition layer. It reads the complete document universe produced by notebook 02, applies the explicit `relevance_flag` already stored on each document row, selects preferred language/format representations, and downloads the corresponding InfoCuria source files.

No case/proceeding/appeal structure is created here. The inclusion rule is therefore visible in the document metadata before acquisition, rather than existing only inside downloader code.

## Inputs and outputs

**Input**

- `output/court_infocuria/court_documents.xlsx` (CSV fallback supported).

**Outputs**

- `output/court_infocuria/court_document_downloads.xlsx` / `.csv` — one row per selected download target, including selection fields, status, attempted/working URL and local raw-file handoff.
- `data/raw/court_infocuria/html/` and `data/raw/court_infocuria/pdf/` — downloaded source files.
- `logs/court_infocuria/document_download_state.jsonl` — resumable working state.

The manifest retains failures and missing files so acquisition completeness can be reported in the Data Descriptor.

In [1]:
from __future__ import annotations

import json
import os
import re
import time
import random
import logging
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("court_infocuria_downloads")

## 1. Paths and acquisition settings

Language and format preferences are explicit here because they belong to acquisition rather than to the source document universe. The default ordering prefers English, then German, then French, and HTML before PDF when the representation is available.

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "02_court").exists():
            return candidate
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the ec_cjeu project root.")

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "output" / "court_infocuria"
LOG_DIR = PROJECT_ROOT / "logs" / "court_infocuria"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "court_infocuria"
HTML_DIR = RAW_DIR / "html"
PDF_DIR = RAW_DIR / "pdf"
for directory in [OUTPUT_DIR, LOG_DIR, HTML_DIR, PDF_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DOCUMENT_METADATA_XLSX = OUTPUT_DIR / "court_documents.xlsx"
DOCUMENT_METADATA_CSV = OUTPUT_DIR / "court_documents.csv"
DOWNLOAD_MANIFEST_XLSX = OUTPUT_DIR / "court_document_downloads.xlsx"
DOWNLOAD_MANIFEST_CSV = OUTPUT_DIR / "court_document_downloads.csv"
DOWNLOAD_MANIFEST_JSONL = LOG_DIR / "document_download_state.jsonl"

RUN_DOWNLOADS = True
OVERWRITE_EXISTING_FILES = False
DOWNLOAD_LIMIT = None
DOWNLOAD_SAVE_EVERY = 100
PREFERRED_LANGUAGES = ["EN", "DE", "FR"]
PREFERRED_FORMATS = ["HTML", "PDF"]
REQUEST_TIMEOUT_SECONDS = 90
DOWNLOAD_SLEEP_SECONDS = 1.25
DOWNLOAD_SLEEP_JITTER_SECONDS = 0.75
RETRIES_PER_URL = 3
RETRY_BACKOFF_SECONDS = 4.0
SESSION_WARMUP = True
SESSION_WARMUP_SLEEP_SECONDS = 0.75
DROP_HUGE_DOWNLOAD_COLUMNS = ["candidate_download_records", "all_candidate_download_urls"]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/126 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml,application/pdf,application/json;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,de;q=0.8,fr;q=0.7",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Origin": "https://infocuria.curia.europa.eu",
    "Referer": "https://infocuria.curia.europa.eu/",
}

COURT_INFOCURIA_RAW_DIR = RAW_DIR
COURT_INFOCURIA_HTML_DIR = HTML_DIR
COURT_INFOCURIA_PDFS_DIR = PDF_DIR
COURT_INFOCURIA_HTML_EXISTING_DIRS = [HTML_DIR]
COURT_INFOCURIA_PDF_EXISTING_DIRS = [PDF_DIR]


def read_manifest(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, dtype=str, keep_default_na=False)
    return pd.read_csv(path, dtype=str, keep_default_na=False, low_memory=False)


def save_jsonl(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(path, orient="records", lines=True, force_ascii=False)


def clean_str(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()


def first_nonempty(*values):
    for value in values:
        s = clean_str(value)
        if s and s.lower() not in {"nan", "none", "<na>"}:
            return s
    return ""


def path_exists_safe(*values):
    s = first_nonempty(*values)
    if not s:
        return False
    try:
        return Path(s).exists()
    except Exception:
        return False


def normalize_doc_formats(value):
    if value is None:
        return []
    if isinstance(value, list):
        vals = value
    elif isinstance(value, str):
        try:
            parsed = json.loads(value)
            vals = parsed if isinstance(parsed, list) else [value]
        except Exception:
            vals = re.split(r"[|,; ]+", value)
    else:
        vals = [value]
    return sorted(set([str(x).upper().strip() for x in vals if str(x).strip()]))


def repo_path(value):
    if value is None:
        return ""
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "<na>"}:
        return ""
    path = Path(text)
    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()

## 2. Read the document universe and select acquisition targets

Notebook 02 already records the project-specific document definition in `relevance_flag`; notebook 03 does not reclassify legal document types. Every relevant row becomes a download target. The manifest additionally records the selection rule and preference ordering so the acquisition criteria are visible as data.

In [3]:
source_path = DOCUMENT_METADATA_XLSX if DOCUMENT_METADATA_XLSX.exists() else DOCUMENT_METADATA_CSV
if not source_path.exists():
    raise FileNotFoundError("Run notebook 02 first; court_documents is missing.")

df_documents = read_manifest(source_path)
if df_documents.empty:
    raise ValueError("court_documents is empty.")
if "relevance_flag" not in df_documents.columns:
    raise KeyError("court_documents must contain relevance_flag from notebook 02.")

relevance = df_documents["relevance_flag"].astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})
df_relevant_docs = df_documents.loc[relevance].copy().reset_index(drop=True)

print(f"All document rows: {len(df_documents):,}")
print(f"Selected relevant document rows: {len(df_relevant_docs):,}")
display(df_relevant_docs[[c for c in ["procedureId", "case_number_raw", "logicDocId", "docType", "normalized_docTypeCode", "available_languages", "docFormats"] if c in df_relevant_docs.columns]].head(20))

All document rows: 154,695
Selected relevant document rows: 42,847


,procedureId,case_number_raw,logicDocId,docType,normalized_docTypeCode,available_languages,docFormats
0,C/0001/00/00000000RD/01/P/01,C-1/00,id_85906,Arrêt (Sommaire),ARRET_SOM,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""PDF""]"
1,C/0001/00/00000000RD/01/P/01,C-1/00,id_46950,Arrêt,ARRET,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML"", ""PDF""]"
2,C/0001/02/00000000RP/01/P/01,C-1/02,id_49061,Arrêt,ARRET,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML"", ""PDF""]"
3,C/0001/02/00000000RP/01/P/01,C-1/02,id_55247,Arrêt (Sommaire),ARRET_SOM,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML"", ""PDF""]"
4,C/0001/03/00000000RP/01/P/01,C-1/03,id_65336,Arrêt (Sommaire),ARRET_SOM,"[""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""NL"", ""PL"", ""PT"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]"
5,C/0001/03/00000000RP/01/P/01,C-1/03,id_49453,Arrêt,ARRET,"[""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""MT"", ""NL"", ""PL"", ""PT"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]"
6,C/0001/04/00000000RP/01/P/01,C-1/04,id_57812,Arrêt,ARRET,"[""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""MT"", ""NL"", ""PL"", ""PT"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]"
7,C/0001/04/00000000RP/01/P/01,C-1/04,id_64087,Arrêt (Sommaire),ARRET_SOM,"[""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""NL"", ""PL"", ""PT"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]"
8,C/0001/05/00000000RP/01/P/01,C-1/05,id_61266,Arrêt (Sommaire),ARRET_SOM,"[""BG"", ""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""NL"", ""PL"", ""PT"", ""RO"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]"
9,C/0001/05/00000000RP/01/P/01,C-1/05,id_64718,Arrêt,ARRET,"[""BG"", ""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""MT"", ""NL"", ""PL"", ""PT"", ""RO"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]"


## 3. Build the download manifest and retrieve source files

For each selected document, candidate URLs are generated from the available languages and formats. Existing local files are reused. Failed attempts remain in the manifest and JSONL state, allowing interrupted runs to resume and acquisition failure rates to be reported explicitly.

In [4]:
# ---------------------------------------------------------------------
# 5) Download URL and target-path builders
# ---------------------------------------------------------------------
# when downloading, but the final manifest stores only attempted URLs, the selected
# first URL, and the final working URL/status/path.

def parse_json_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            obj = json.loads(x)
            return obj if isinstance(obj, list) else []
        except Exception:
            return []
    return []


def ordered_unique(values):
    out = []
    seen = set()
    for v in values:
        s = clean_str(v)
        if s and s not in seen:
            seen.add(s)
            out.append(s)
    return out


def available_languages_for_row(row):
    langs = [str(x).upper() for x in parse_json_list(row.get("available_languages")) if x]
    proc_lang = clean_str(row.get("procLang")).upper()
    ordered = []
    for lang in PREFERRED_LANGUAGES:
        if lang.upper() in langs:
            ordered.append(lang.upper())
    if proc_lang and proc_lang in langs:
        ordered.append(proc_lang)
    ordered.extend(langs)
    return ordered_unique(ordered)


def available_formats_for_row(row):
    fmts = [str(x).upper() for x in parse_json_list(row.get("docFormats")) if x]
    if not fmts and clean_str(row.get("docFormat")):
        fmts = normalize_doc_formats(row.get("docFormat"))
    ordered = []
    for fmt in PREFERRED_FORMATS:
        if fmt.upper() in fmts:
            ordered.append(fmt.upper())
    ordered.extend(fmts)
    return ordered_unique(ordered)


def language_format_pairs_for_row(row):
    langs = available_languages_for_row(row) or ["EN"]
    fmts = available_formats_for_row(row) or ["HTML", "PDF"]
    pairs = []
    for lang in langs:
        for fmt in fmts:
            pairs.append((lang, fmt))
    return pairs


def year_from_date_value(value):
    s = clean_str(value)
    if not s:
        return ""
    m = re.search(r"(19|20)\d{2}", s)
    if m:
        return m.group(0)
    for dayfirst in [False, True]:
        try:
            dt = pd.to_datetime(s, errors="coerce", dayfirst=dayfirst)
            if pd.notna(dt):
                return f"{int(dt.year):04d}"
        except Exception:
            pass
    return ""


def parse_routing_parts(routing):
    parts = str(routing or "").strip().split("/")
    if len(parts) < 3:
        return None
    court, number, two_digit_year, *rest = parts
    if not court or not number or not two_digit_year:
        return None
    return court, number, two_digit_year, rest


def full_year_from_two_digit(two_digit_year):
    try:
        y = int(str(two_digit_year))
        return "19" + f"{y:02d}" if y >= 50 else "20" + f"{y:02d}"
    except Exception:
        return ""


def routing_case_year(routing):
    parsed = parse_routing_parts(routing)
    return full_year_from_two_digit(parsed[2]) if parsed else ""


def candidate_years_for_row(row):
    routing = first_nonempty(row.get("idProcedure"), row.get("procedureId"))
    base_years = []
    for y in [
        routing_case_year(routing),
        year_from_date_value(row.get("docDate")),
        year_from_date_value(row.get("closeDate")),
        year_from_date_value(row.get("procClosDate")),
        year_from_date_value(row.get("introductionDate")),
        year_from_date_value(row.get("procIntroDate")),
        clean_str(row.get("case_year_full")),
    ]:
        if re.fullmatch(r"(19|20)\d{2}", clean_str(y)):
            base_years.append(int(y))
    if not base_years:
        return []
    lo, hi = min(base_years), max(base_years)
    years = list(range(lo - 2, hi + 3))
    ordered = [int(y) for y in base_years] + years
    return [str(y) for y in ordered_unique([str(y) for y in ordered if 1950 <= int(y) <= 2035])]


def routing_to_blob_path(routing, override_year=None):
    parsed = parse_routing_parts(routing)
    if not parsed:
        return None
    court, number, two_digit_year, rest = parsed
    full_year = clean_str(override_year) or full_year_from_two_digit(two_digit_year)
    if not full_year:
        return None
    filename_part = f"{court}_{number}_{two_digit_year}_" + "_".join(rest)
    return f"{court}/{full_year}/{filename_part}"


def build_download_url(row, lang, fmt, override_year=None):
    logic_doc_id = clean_str(row.get("logicDocId"))
    routing = first_nonempty(row.get("idProcedure"), row.get("procedureId"))
    if not logic_doc_id or not routing or not lang or not fmt:
        return ""
    blob_path = routing_to_blob_path(routing, override_year=override_year)
    if not blob_path:
        return ""
    logic_num = logic_doc_id.replace("id_", "")
    fmt = str(fmt).upper()
    lang = str(lang).upper()
    if fmt == "HTML":
        return f"https://infocuriaws.curia.europa.eu/blob/download-file-html/{blob_path}/{logic_num}-{lang}-1.html"
    if fmt == "PDF":
        return f"https://infocuriaws.curia.europa.eu/blob/download-file/{blob_path}/{logic_num}-{lang}-1.pdf"
    return ""


def safe_filename(value, fallback="document"):
    value = "" if value is None else str(value)
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("_")
    return value or fallback


def filename_from_url(url, fallback):
    try:
        parsed = urlparse(str(url))
        name = Path(parsed.path).name
        return safe_filename(name, fallback=fallback)
    except Exception:
        return safe_filename(fallback)


def primary_target_dir_for_format(fmt):
    fmt = str(fmt or "").upper()
    if fmt == "HTML":
        return COURT_INFOCURIA_HTML_DIR
    if fmt == "PDF":
        return COURT_INFOCURIA_PDFS_DIR
    return COURT_INFOCURIA_RAW_DIR / "other"


def candidate_existing_dirs_for_format(fmt):
    fmt = str(fmt or "").upper()
    if fmt == "HTML":
        return COURT_INFOCURIA_HTML_EXISTING_DIRS
    if fmt == "PDF":
        return COURT_INFOCURIA_PDF_EXISTING_DIRS
    return [COURT_INFOCURIA_RAW_DIR / "other"]


def find_existing_download_path(fmt, filename):
    for folder in candidate_existing_dirs_for_format(fmt):
        candidate = Path(folder) / filename
        if candidate.exists():
            return candidate.resolve()
    return None


def build_candidate_download_records(row):
    records = []
    years = candidate_years_for_row(row) or [None]
    for lang, fmt in language_format_pairs_for_row(row):
        for year in years:
            url = build_download_url(row, lang, fmt, override_year=year)
            if not url:
                continue
            fallback_base = safe_filename("_".join([
                clean_str(row.get("procedure_key")),
                first_nonempty(row.get("logicDocId"), row.get("docId")),
                clean_str(row.get("normalized_docTypeCode")),
                clean_str(lang),
                clean_str(year),
            ]), fallback="infocuria_document")
            ext = "html" if fmt == "HTML" else "pdf" if fmt == "PDF" else "bin"
            filename = filename_from_url(url, fallback=f"{fallback_base}.{ext}")
            target_dir = primary_target_dir_for_format(fmt)
            target_dir.mkdir(parents=True, exist_ok=True)
            intended_local_path = target_dir / filename
            existing_path = find_existing_download_path(fmt, filename)
            records.append({
                "url": url,
                "language": lang,
                "format": fmt,
                "year": year,
                "filename": filename,
                "local_path": str(existing_path if existing_path is not None else intended_local_path),
                "intended_local_path": str(intended_local_path),
                "existing_local_path": str(existing_path) if existing_path is not None else "",
                "file_exists_before_download": bool(existing_path is not None or intended_local_path.exists()),
            })
    out = []
    seen = set()
    for r in records:
        if r["url"] not in seen:
            seen.add(r["url"])
            out.append(r)
    return out


def build_download_manifest(relevant_docs):
    """Build one download target row per relevant document.

    Every row in df_relevant_docs, including ORD_COMM and
    combined codes such as ORD_COMM|REQ_COMM, receives a download-manifest row.
    Candidate URL lists are generated only in memory and are not exported.
    """
    rows = []
    for _, row in relevant_docs.iterrows():
        candidate_records = build_candidate_download_records(row)
        first = candidate_records[0] if candidate_records else {}
        already_exists = bool(first.get("file_exists_before_download", False))
        rows.append({
            "internal_key": row.get("internal_key"),
            "procedure_key": row.get("procedure_key"),
            "procedure_key_base": row.get("procedure_key_base"),
            "case_number_raw": row.get("case_number_raw"),
            "case_number_clean": row.get("case_number_clean"),
            "affId": row.get("affId"),
            "procedureId": row.get("procedureId"),
            "publishedId": row.get("publishedId"),
            "docId": row.get("docId"),
            "logicDocId": row.get("logicDocId"),
            "jpLogicDocId": row.get("jpLogicDocId"),
            "idProcedure": row.get("idProcedure"),
            "idPublished": row.get("idPublished"),
            "docType": row.get("docType"),
            "docTypeCode": row.get("docTypeCode"),
            "normalized_docTypeCode": row.get("normalized_docTypeCode"),
            "docDate": row.get("docDate"),
            "available_languages": row.get("available_languages"),
            "docFormats": row.get("docFormats"),
            "candidate_years": json.dumps(candidate_years_for_row(row), ensure_ascii=False),
            "download_url": first.get("url", ""),
            "selected_language": first.get("language", ""),
            "selected_format": first.get("format", ""),
            "selected_year": first.get("year", ""),
            "target_filename": first.get("filename", ""),
            "intended_local_path": first.get("intended_local_path", ""),
            "existing_local_path": first.get("existing_local_path", ""),
            "final_local_path": first.get("local_path", "") if already_exists else pd.NA,
            "file_present_on_disk": already_exists,
            "file_exists_before_download": already_exists,
            "download_success": already_exists,
            "download_status": "already_exists" if already_exists else "pending",
            "attempted_urls": json.dumps([], ensure_ascii=False),
            "attempted_http_statuses": json.dumps([], ensure_ascii=False),
            "attempt_count": 0,
            "working_url": first.get("url", "") if already_exists else pd.NA,
            "working_language": first.get("language", "") if already_exists else pd.NA,
            "working_format": first.get("format", "") if already_exists else pd.NA,
            "working_year": first.get("year", "") if already_exists else pd.NA,
            "http_status": pd.NA,
            "error": pd.NA,
            "bytes": pd.NA,
        })
    df = pd.DataFrame(rows)
    return load_previous_download_status_if_available(df)


def load_previous_download_status_if_available(df):
    if df.empty or not DOWNLOAD_MANIFEST_JSONL.exists():
        return df
    try:
        prev = read_manifest(DOWNLOAD_MANIFEST_JSONL)
    except Exception as e:
        print(f"Could not load previous download manifest for status merge: {e}")
        return df
    key_cols = [c for c in ["logicDocId", "idProcedure"] if c in df.columns and c in prev.columns]
    if not key_cols:
        key_cols = [c for c in ["docId", "procedureId"] if c in df.columns and c in prev.columns]
    status_cols = [c for c in [
        "download_success", "download_status", "http_status", "error", "bytes", "working_url",
        "working_language", "working_format", "working_year", "final_local_path",
        "attempted_urls", "attempted_http_statuses", "attempt_count"
    ] if c in prev.columns]
    if not key_cols or not status_cols:
        return df
    prev_small = prev[key_cols + status_cols].drop_duplicates(subset=key_cols, keep="last")
    out = df.merge(prev_small, on=key_cols, how="left", suffixes=("", "__previous"))
    for col in status_cols:
        prev_col = f"{col}__previous"
        if prev_col in out.columns:
            has_prev = out[prev_col].notna() & out[prev_col].astype(str).ne("")
            out.loc[has_prev, col] = out.loc[has_prev, prev_col]
            out = out.drop(columns=[prev_col])
    return out


df_download_manifest = build_download_manifest(df_relevant_docs)
df_download_manifest = df_download_manifest.drop(columns=[c for c in DROP_HUGE_DOWNLOAD_COLUMNS + ["document_dedup_key"] if c in df_download_manifest.columns], errors="ignore")
def _document_identity_frame(df):
    key_cols = [c for c in ["logicDocId", "docId", "procedureId", "idProcedure", "affId"] if c in df.columns]
    return df[key_cols].astype(str).replace({"nan": "", "<NA>": ""}) if key_cols else pd.DataFrame(index=df.index)

if not df_relevant_docs.empty:
    rel_key_cols = [c for c in ["logicDocId", "docId", "procedureId", "idProcedure", "affId"] if c in df_relevant_docs.columns and c in df_download_manifest.columns]
    if rel_key_cols:
        rel_keys = df_relevant_docs[rel_key_cols].astype(str).replace({"nan": "", "<NA>": ""}).drop_duplicates()
        dl_keys = df_download_manifest[rel_key_cols].astype(str).replace({"nan": "", "<NA>": ""}).drop_duplicates()
        missing_targets = rel_keys.merge(dl_keys, on=rel_key_cols, how="left", indicator=True)
        missing_targets = missing_targets[missing_targets["_merge"].eq("left_only")].drop(columns=["_merge"])
        print("Relevant document rows targeted for download:", f"{len(df_download_manifest):,} / {len(df_relevant_docs):,}")
        print("Relevant document keys missing from download manifest:", f"{len(missing_targets):,}")
        if len(missing_targets):
            display(missing_targets.head(20))
    if "normalized_docTypeCode" in df_relevant_docs.columns:
        print("Relevant document types targeted for download, top 50:")
        display(df_relevant_docs["normalized_docTypeCode"].fillna("UNKNOWN").value_counts().head(50).to_frame("relevant_download_targets"))

print("Download manifest rows:", len(df_download_manifest))
if not df_download_manifest.empty:
    print("Rows with first candidate URL:", df_download_manifest["download_url"].fillna("").ne("").sum())
    print("Rows already existing:", df_download_manifest["file_exists_before_download"].fillna(False).astype(bool).sum())
    display(df_download_manifest[[
        c for c in ["internal_key", "procedure_key", "publishedId", "logicDocId", "normalized_docTypeCode", "docDate", "selected_language", "selected_format", "selected_year", "download_url", "intended_local_path", "existing_local_path", "download_status"]
        if c in df_download_manifest.columns
    ]].head(20))

# ---------------------------------------------------------------------
# 6) Optional downloader for relevant documents only
# ---------------------------------------------------------------------

SUCCESS_STATUSES = {"downloaded", "already_exists"}
FAILED_OR_RETRYABLE_STATUSES = {
    "pending", "failed_http", "failed_exception", "failed_missing_url", "failed_missing_target_path", "failed_empty_content",
}


def candidate_records_from_row(row):
    """Generate candidate download records in memory.

    Candidate URL lists are rebuilt from metadata instead of being stored in the manifest
    columns in the manifest. The candidates are rebuilt from metadata when needed.
    """
    records = build_candidate_download_records(row)
    if records:
        return records
    if clean_str(row.get("download_url")):
        return [{
            "url": row.get("download_url"),
            "language": row.get("selected_language"),
            "format": row.get("selected_format"),
            "year": row.get("selected_year"),
            "local_path": first_nonempty(row.get("final_local_path"), row.get("existing_local_path"), row.get("intended_local_path")),
        }]
    return []


def infer_tabs_document_url(file_url, row=None):
    try:
        parsed = urlparse(str(file_url))
        parts = parsed.path.strip("/").split("/")
        if len(parts) < 4:
            return "https://infocuria.curia.europa.eu/"
        filename = parts[-1]
        logic_fmt = filename.replace(".", "-")
        doc_type = first_nonempty((row or {}).get("normalized_docTypeCode"), (row or {}).get("docTypeCode"))
        routing = first_nonempty((row or {}).get("idProcedure"), (row or {}).get("procedureId"))
        if routing and doc_type:
            routing_pretty = routing.replace("/", "-")
            return f"https://infocuria.curia.europa.eu/tabs/document/{routing_pretty}/{doc_type}/{logic_fmt}"
    except Exception:
        pass
    return "https://infocuria.curia.europa.eu/"


def get_download_session():
    session = requests.Session()
    session.headers.update(HEADERS)
    return session


def polite_sleep(base=None):
    base = DOWNLOAD_SLEEP_SECONDS if base is None else base
    delay = float(base) + random.uniform(0, float(DOWNLOAD_SLEEP_JITTER_SECONDS))
    if delay > 0:
        time.sleep(delay)


def warmup_session_for_url(session, file_url, row):
    if not SESSION_WARMUP:
        return None
    referrer = infer_tabs_document_url(file_url, row=row)
    try:
        session.get(referrer, timeout=REQUEST_TIMEOUT_SECONDS, headers={**HEADERS, "Referer": "https://infocuria.curia.europa.eu/"}, allow_redirects=True)
        time.sleep(float(SESSION_WARMUP_SLEEP_SECONDS))
        return referrer
    except Exception:
        return "https://infocuria.curia.europa.eu/"


def download_one_file(row, overwrite=False, session=None):
    records = candidate_records_from_row(row)
    if not records:
        return {
            "download_success": False,
            "download_status": "failed_missing_url",
            "http_status": pd.NA,
            "error": "Missing candidate_download_records/download_url",
            "bytes": pd.NA,
            "working_url": pd.NA,
            "working_language": pd.NA,
            "working_format": pd.NA,
            "working_year": pd.NA,
            "final_local_path": pd.NA,
            "attempted_urls": json.dumps([], ensure_ascii=False),
            "attempted_http_statuses": json.dumps([], ensure_ascii=False),
            "attempt_count": 0,
        }

    session = session or get_download_session()
    attempted_urls = []
    attempted_statuses = []
    last_error = ""
    last_status = pd.NA
    last_length = 0

    for record in records:
        file_url = clean_str(record.get("url"))
        target_path = Path(clean_str(record.get("local_path"))) if clean_str(record.get("local_path")) else None
        selected_format = clean_str(record.get("format")).upper()

        if not file_url or target_path is None:
            continue

        if target_path.exists() and not overwrite:
            return {
                "download_success": True,
                "download_status": "already_exists",
                "http_status": pd.NA,
                "error": pd.NA,
                "bytes": target_path.stat().st_size,
                "working_url": file_url,
                "working_language": record.get("language"),
                "working_format": selected_format,
                "working_year": record.get("year"),
                "final_local_path": str(target_path),
                "attempted_urls": json.dumps(attempted_urls, ensure_ascii=False),
                "attempted_http_statuses": json.dumps(attempted_statuses, ensure_ascii=False),
                "attempt_count": len(attempted_urls),
            }

        referrer = warmup_session_for_url(session, file_url, row)

        for retry_no in range(1, int(RETRIES_PER_URL) + 1):
            attempted_urls.append(file_url)
            try:
                response = session.get(
                    file_url,
                    timeout=REQUEST_TIMEOUT_SECONDS,
                    headers={**HEADERS, "Referer": referrer or "https://infocuria.curia.europa.eu/"},
                    allow_redirects=True,
                )
                status_code = response.status_code
                content = response.content or b""
                last_status = status_code
                last_length = len(content)
                attempted_statuses.append(status_code)

                if status_code == 200 and content:
                    target_path.parent.mkdir(parents=True, exist_ok=True)
                    if selected_format == "HTML" or file_url.lower().endswith(".html"):
                        target_path.write_text(response.text, encoding="utf-8")
                    else:
                        target_path.write_bytes(content)
                    return {
                        "download_success": True,
                        "download_status": "downloaded",
                        "http_status": status_code,
                        "error": pd.NA,
                        "bytes": target_path.stat().st_size,
                        "working_url": file_url,
                        "working_language": record.get("language"),
                        "working_format": selected_format,
                        "working_year": record.get("year"),
                        "final_local_path": str(target_path),
                        "attempted_urls": json.dumps(attempted_urls, ensure_ascii=False),
                        "attempted_http_statuses": json.dumps(attempted_statuses, ensure_ascii=False),
                        "attempt_count": len(attempted_urls),
                    }

                last_error = f"HTTP {status_code}; length={len(content)}"
                if status_code in {403, 429, 500, 502, 503, 504} and retry_no < int(RETRIES_PER_URL):
                    time.sleep(float(RETRY_BACKOFF_SECONDS) * retry_no + random.uniform(0, 1.5))
                    continue
                # 404: try the next URL candidate immediately.
                break
            except Exception as e:
                last_error = repr(e)
                attempted_statuses.append("EXCEPTION")
                if retry_no < int(RETRIES_PER_URL):
                    time.sleep(float(RETRY_BACKOFF_SECONDS) * retry_no + random.uniform(0, 1.5))
                    continue
                break
        polite_sleep()

    return {
        "download_success": False,
        "download_status": "failed_http" if attempted_urls else "failed_missing_url",
        "http_status": last_status,
        "error": f"{last_error}; last_length={last_length}",
        "bytes": last_length,
        "working_url": pd.NA,
        "working_language": pd.NA,
        "working_format": pd.NA,
        "working_year": pd.NA,
        "final_local_path": pd.NA,
        "attempted_urls": json.dumps(attempted_urls, ensure_ascii=False),
        "attempted_http_statuses": json.dumps(attempted_statuses, ensure_ascii=False),
        "attempt_count": len(attempted_urls),
    }


def run_downloads(df_download, limit=None, overwrite=False, save_every=100):
    df = df_download.copy()
    if df.empty:
        print("Download manifest is empty.")
        return df

    # If a final/specified local path exists, mark it as already present.
    # Use NA-safe helpers: pandas pd.NA cannot be used with `or`/`if`.
    existing_now = df.apply(
        lambda r: path_exists_safe(r.get("final_local_path"), r.get("existing_local_path"), r.get("intended_local_path")),
        axis=1,
    )
    if not overwrite:
        df.loc[existing_now, "download_success"] = True
        df.loc[existing_now, "download_status"] = "already_exists"

    status = df["download_status"].fillna("pending").astype(str)
    success_bool = df.get("download_success", pd.Series(False, index=df.index)).astype(str).str.lower().isin(["true", "1", "yes"])
    has_candidates = df["download_url"].fillna("").astype(str).str.strip().ne("")

    needs_attempt = (
        has_candidates
        & (overwrite | ~existing_now)
        & (
            overwrite
            | status.isin(FAILED_OR_RETRYABLE_STATUSES)
            | status.str.startswith("failed")
            | (~success_bool & ~status.isin(SUCCESS_STATUSES))
        )
        & ~status.isin(SUCCESS_STATUSES)
    )

    pending_indices = df.index[needs_attempt].tolist()
    if limit is not None:
        pending_indices = pending_indices[:int(limit)]

    print(f"Download attempts planned: {len(pending_indices):,}")
    session = get_download_session()
    for n, idx in enumerate(tqdm(pending_indices, desc="Downloading relevant InfoCuria documents"), start=1):
        result = download_one_file(df.loc[idx].to_dict(), overwrite=overwrite, session=session)
        for k, v in result.items():
            df.at[idx, k] = v
        if save_every and n % int(save_every) == 0:
            save_jsonl(df.drop(columns=[c for c in DROP_HUGE_DOWNLOAD_COLUMNS + ["document_dedup_key"] if c in df.columns], errors="ignore"), DOWNLOAD_MANIFEST_JSONL)

    df = df.drop(columns=[c for c in DROP_HUGE_DOWNLOAD_COLUMNS + ["document_dedup_key"] if c in df.columns], errors="ignore")
    save_jsonl(df, DOWNLOAD_MANIFEST_JSONL)
    return df


if RUN_DOWNLOADS:
    df_download_manifest = run_downloads(
        df_download_manifest,
        limit=DOWNLOAD_LIMIT,
        overwrite=OVERWRITE_EXISTING_FILES,
        save_every=DOWNLOAD_SAVE_EVERY,
    )
else:
    print("RUN_DOWNLOADS=False. No documents downloaded.")
    print("The download manifest contains selected/attempted URLs and previous/current status where available.")

Could not load previous download manifest for status merge: Error tokenizing data. C error: Expected 58 fields in line 3, saw 59

Relevant document rows targeted for download: 42,847 / 42,847
Relevant document keys missing from download manifest: 0
Relevant document types targeted for download, top 50:


,relevant_download_targets
normalized_docTypeCode,
ARRET,19126
ARRET_SOM,16866
ARRET_DR,6836
DECISION,18
RECT,1


Download manifest rows: 42847
Rows with first candidate URL: 42847
Rows already existing: 42501


,internal_key,procedure_key,publishedId,logicDocId,normalized_docTypeCode,docDate,selected_language,selected_format,selected_year,download_url,intended_local_path,existing_local_path,download_status
0,2000/0001/C//,2000/0001/C//,C-1/00,id_85906,ARRET_SOM,2001-12-13,EN,PDF,2000,https://infocuriaws.curia.europa.eu/blob/download-file/C/2000/C_0001_00_00000000RD_01_P_01/85906-EN-1.pdf,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/pdf/85906-EN-1.pdf,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/pdf/85906-EN-1.pdf,already_exists
1,2000/0001/C//,2000/0001/C//,C-1/00,id_46950,ARRET,2001-12-13,EN,HTML,2000,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2000/C_0001_00_00000000RD_01_P_01/46950-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/46950-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/46950-EN-1.html,already_exists
2,2002/0001/C//,2002/0001/C//,C-1/02,id_49061,ARRET,2004-04-01,EN,HTML,2002,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2002/C_0001_02_00000000RP_01_P_01/49061-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/49061-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/49061-EN-1.html,already_exists
3,2002/0001/C//,2002/0001/C//,C-1/02,id_55247,ARRET_SOM,2004-04-01,EN,HTML,2002,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2002/C_0001_02_00000000RP_01_P_01/55247-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/55247-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/55247-EN-1.html,already_exists
4,2003/0001/C//,2003/0001/C//,C-1/03,id_65336,ARRET_SOM,2004-09-07,EN,HTML,2003,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2003/C_0001_03_00000000RP_01_P_01/65336-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/65336-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/65336-EN-1.html,already_exists
5,2003/0001/C//,2003/0001/C//,C-1/03,id_49453,ARRET,2004-09-07,EN,HTML,2003,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2003/C_0001_03_00000000RP_01_P_01/49453-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/49453-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/49453-EN-1.html,already_exists
6,2004/0001/C//,2004/0001/C//,C-1/04,id_57812,ARRET,2006-01-17,EN,HTML,2004,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2004/C_0001_04_00000000RP_01_P_01/57812-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/57812-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/57812-EN-1.html,already_exists
7,2004/0001/C//,2004/0001/C//,C-1/04,id_64087,ARRET_SOM,2006-01-17,EN,HTML,2004,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2004/C_0001_04_00000000RP_01_P_01/64087-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/64087-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/64087-EN-1.html,already_exists
8,2005/0001/C//,2005/0001/C//,C-1/05,id_61266,ARRET_SOM,2007-01-09,EN,HTML,2005,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2005/C_0001_05_00000000RP_01_P_01/61266-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/61266-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/61266-EN-1.html,already_exists
9,2005/0001/C//,2005/0001/C//,C-1/05,id_64718,ARRET,2007-01-09,EN,HTML,2005,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2005/C_0001_05_00000000RP_01_P_01/64718-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/64718-EN-1.html,/home/edik/projects/ec_cjeu/data/raw/court_infocuria/html/64718-EN-1.html,already_exists


Download attempts planned: 346


## 4. Finalize and save the download manifest

The final handoff exposes the actual raw file name/path/format used by notebook 04. Selection criteria are also repeated as columns so a repository user can understand why the row entered the acquisition queue without reading Python source.

In [5]:
def add_raw_file_handoff(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    raw_paths, raw_names, raw_formats, success = [], [], [], []
    for _, row in out.iterrows():
        chosen = first_nonempty(row.get("final_local_path"), row.get("existing_local_path"), row.get("intended_local_path"))
        path = Path(chosen) if chosen else None
        exists = bool(path and path.exists() and path.is_file())
        raw_paths.append(repo_path(path) if exists else "")
        raw_names.append(path.name if exists else "")
        raw_formats.append(path.suffix.lower().lstrip(".") if exists else clean_str(row.get("working_format")).lower())
        success.append(bool(exists or str(row.get("download_success", "")).lower() in {"true", "1", "yes"}))
    out["success"] = success
    out["raw_file_name"] = raw_names
    out["raw_file_path"] = raw_paths
    out["file_format"] = raw_formats
    out["selected_for_download"] = True
    out["selection_rule"] = "relevance_flag == True"
    out["language_preference"] = " > ".join(PREFERRED_LANGUAGES)
    out["format_preference"] = " > ".join(PREFERRED_FORMATS)
    return out


df_download_manifest = add_raw_file_handoff(df_download_manifest)
df_download_manifest = df_download_manifest.drop(
    columns=[c for c in DROP_HUGE_DOWNLOAD_COLUMNS + ["document_dedup_key"] if c in df_download_manifest.columns],
    errors="ignore",
)

df_download_manifest.to_excel(DOWNLOAD_MANIFEST_XLSX, index=False, sheet_name="downloads")
df_download_manifest.to_csv(DOWNLOAD_MANIFEST_CSV, index=False)

print("Saved download manifest:", DOWNLOAD_MANIFEST_XLSX)
print("Rows:", f"{len(df_download_manifest):,}")

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Saved download manifest: /home/edik/projects/ec_cjeu/output/court_infocuria/court_document_downloads.xlsx
Rows: 42,847


## 5. Technical validation summary

Report target counts, successful raw-file handoffs, failures, and the language/format actually selected. These quantities are suitable for acquisition-completeness reporting in the Data Descriptor.

In [6]:
print(f"Download targets: {len(df_download_manifest):,}")
print("Successful raw-file handoffs:", int(df_download_manifest["raw_file_path"].fillna("").astype(str).str.strip().ne("").sum()))
if "download_status" in df_download_manifest.columns:
    display(df_download_manifest["download_status"].fillna("missing").value_counts(dropna=False).rename_axis("download_status").reset_index(name="rows"))
if "selected_language" in df_download_manifest.columns:
    display(df_download_manifest["selected_language"].fillna("missing").value_counts().rename_axis("language").reset_index(name="rows"))
if "selected_format" in df_download_manifest.columns:
    display(df_download_manifest["selected_format"].fillna("missing").value_counts().rename_axis("format").reset_index(name="rows"))

Download targets: 42,847
Successful raw-file handoffs: 42847


,download_status,rows
0,already_exists,42847


,language,rows
0,EN,40363
1,FR,1452
2,DE,862
3,IT,45
4,EL,41
5,DA,30
6,ES,19
7,NL,14
8,PT,8
9,FI,5


,format,rows
0,HTML,30728
1,PDF,12119
